# Basic 2 mT5-Base FOLIO Training

fine-tune `google/mt5-base` on FOLIO for natural language -> FOL generation

# Environment Check And Dependencies

In [1]:
# !pip install -q -U transformers datasets accelerate sentencepiece peft tqdm

import importlib.util

required = ["transformers", "datasets", "accelerate", "sentencepiece", "peft", "tqdm"]
missing = []

for pkg in required:
    ok = importlib.util.find_spec(pkg) is not None
    print(pkg, ok)
    if not ok:
        missing.append(pkg)

print("Missing:", missing)
if missing:
    print("If Internet is OFF, attach a Kaggle dataset/wheel cache containing these packages.")

transformers True
datasets True
accelerate True
sentencepiece True
peft True
tqdm True
Missing: []


In [2]:
import os
import json
import math
import random
import inspect
from pathlib import Path
from typing import Any, Dict, List, Sequence

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [5]:
# ============================================================
# Resolve official Basic 2 FOLIO dataset only
# Target dataset: /kaggle/input/folio-dataset
# ============================================================

def load_json_list(path: Path) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON list.")
    return data


def first_present(item: Dict[str, Any], keys: Sequence[str]) -> Any:
    for key in keys:
        if key in item:
            return item[key]
    return None


def count_supervised_fol_rows(path: Path) -> int:
    rows = load_json_list(path)
    good = 0

    for row in rows:
        if not isinstance(row, dict):
            continue

        nat_premises = first_present(row, ("nat_premises", "nat_premise", "premises"))
        nat_conclusion = first_present(row, ("nat_conclusion", "conclusion"))
        fol_premises = first_present(row, ("fol_premises", "fol_premise"))
        fol_conclusion = row.get("fol_conclusion")

        if nat_premises and nat_conclusion and fol_premises and fol_conclusion:
            good += 1

    return good


DATA_ROOT = Path("/kaggle/input/datasets/trnlnhtho/folio-dataset")

TRAIN_PATH = DATA_ROOT / "folio_train.json"
VALID_PATH = DATA_ROOT / "folio_valid.json"
TEST_PATH = DATA_ROOT / "manual_test.json"

USE_INTERNAL_SPLIT = False

print("Using official folio-dataset only.")
print("DATA_ROOT:", DATA_ROOT)

for name, path in [
    ("TRAIN", TRAIN_PATH),
    ("VALID", VALID_PATH),
    ("TEST", TEST_PATH),
]:
    print("=" * 80)
    print(name)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        raise FileNotFoundError(f"{name} file not found: {path}")

    rows = load_json_list(path)
    supervised_count = count_supervised_fol_rows(path)

    labels = {}
    for row in rows:
        if isinstance(row, dict):
            label = row.get("label")
            if label is not None:
                labels[str(label)] = labels.get(str(label), 0) + 1

    print("Rows:", len(rows))
    print("Supervised FOL rows:", supervised_count)
    print("Labels:", labels)

    if rows and isinstance(rows[0], dict):
        first = rows[0]
        print("First id:", first.get("id") or first.get("story_id"))
        print("First keys:", sorted(first.keys()))

if count_supervised_fol_rows(TRAIN_PATH) == 0:
    raise ValueError("folio_train.json has no supervised FOL rows.")

if count_supervised_fol_rows(VALID_PATH) == 0:
    raise ValueError("folio_valid.json has no supervised FOL rows.")

print("\nResolved official paths:")
print("TRAIN_PATH:", TRAIN_PATH)
print("VALID_PATH:", VALID_PATH)
print("TEST_PATH :", TEST_PATH)
print("USE_INTERNAL_SPLIT:", USE_INTERNAL_SPLIT)

Using official folio-dataset only.
DATA_ROOT: /kaggle/input/datasets/trnlnhtho/folio-dataset
TRAIN
Path: /kaggle/input/datasets/trnlnhtho/folio-dataset/folio_train.json
Exists: True
Rows: 856
Supervised FOL rows: 856
Labels: {'True': 256, 'False': 206, 'Uncertain': 394}
First id: folio_train_0
First keys: ['fol_conclusion', 'fol_premises', 'id', 'label', 'nat_conclusion', 'nat_premises', 'story_id']
VALID
Path: /kaggle/input/datasets/trnlnhtho/folio-dataset/folio_valid.json
Exists: True
Rows: 93
Supervised FOL rows: 93
Labels: {'Uncertain': 34, 'True': 35, 'False': 24}
First id: folio_valid_0
First keys: ['fol_conclusion', 'fol_premises', 'id', 'label', 'nat_conclusion', 'nat_premises', 'story_id']
TEST
Path: /kaggle/input/datasets/trnlnhtho/folio-dataset/manual_test.json
Exists: True
Rows: 600
Supervised FOL rows: 0
Labels: {'Uncertain': 199, 'True': 193, 'False': 208}
First id: manual_0
First keys: ['conclusion', 'id', 'label', 'premises']

Resolved official paths:
TRAIN_PATH: /kaggl

In [6]:
MODEL_CANDIDATES = [
    Path("/kaggle/input/datasets/trnlnhtho/mt5-base/mt5-base"),
    Path("/kaggle/input/mt5-base/mt5-base"),
    Path("/kaggle/input/mt5-base"),
    Path("/kaggle/working/mt5-base"),
]
if Path("/kaggle/input").exists():
    for cfg in Path("/kaggle/input").rglob("config.json"):
        MODEL_CANDIDATES.append(cfg.parent)

MODEL_NAME = None
for p in MODEL_CANDIDATES:
    if (p / "config.json").exists() and (
        (p / "pytorch_model.bin").exists()
        or (p / "model.safetensors").exists()
        or any(p.glob("*.safetensors"))
        or any(p.glob("pytorch_model*.bin"))
    ):
        MODEL_NAME = str(p)
        break
if MODEL_NAME is None:
    raise FileNotFoundError("Could not find local mT5-base folder.")

LOCAL_FILES_ONLY = True
RUN_TAG = "basic2_mt5base_raw_folio_official_lora_qkvo_r32"
OUTPUT_DIR = f"/kaggle/working/outputs/{RUN_TAG}"
MERGED_OUTPUT_DIR = f"/kaggle/working/outputs/{RUN_TAG}_merged"
TARBALL_PATH = f"/kaggle/working/{RUN_TAG}.tar.gz"

RUN_MODE = "full"  # "debug" or "full"
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 512
GEN_MAX_NEW_TOKENS = 256
NUM_EPOCHS_DEBUG = 1
NUM_EPOCHS_FULL = 10
LEARNING_RATE = 3e-5
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 4
SEED = 42
EVAL_STEPS = 25
SAVE_STEPS = 25
LOGGING_STEPS = 5
SAVE_TOTAL_LIMIT = 2

USE_LORA = True
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q", "k", "v", "o"]

ENABLE_GRADIENT_CHECKPOINTING = False
USE_FP16 = False
USE_BF16 = False
OPTIM = "adamw_torch"
SAVE_MERGED_MODEL = False
CREATE_TARBALL = True

USE_INSTRUCTION = True
INSTRUCTION = "Translate the following natural language premises and conclusion to first-order logic."
VALID_RATIO = 0.10

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(MERGED_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("MODEL_NAME:", MODEL_NAME)
print("TRAIN_PATH:", TRAIN_PATH)
print("VALID_PATH:", VALID_PATH)
print("RUN_MODE:", RUN_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("USE_LORA:", USE_LORA)
print("LoRA r/alpha/dropout/targets:", LORA_R, LORA_ALPHA, LORA_DROPOUT, LORA_TARGET_MODULES)
print("Epochs debug/full:", NUM_EPOCHS_DEBUG, NUM_EPOCHS_FULL)
print("LR:", LEARNING_RATE)

MODEL_NAME: /kaggle/input/datasets/trnlnhtho/mt5-base/mt5-base
TRAIN_PATH: /kaggle/input/datasets/trnlnhtho/folio-dataset/folio_train.json
VALID_PATH: /kaggle/input/datasets/trnlnhtho/folio-dataset/folio_valid.json
RUN_MODE: full
OUTPUT_DIR: /kaggle/working/outputs/basic2_mt5base_raw_folio_official_lora_qkvo_r32
USE_LORA: True
LoRA r/alpha/dropout/targets: 32 64 0.05 ['q', 'k', 'v', 'o']
Epochs debug/full: 1 10
LR: 3e-05


In [8]:
def as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        return "\n".join(str(item).strip() for item in value if str(item).strip())
    return str(value).strip()


def normalize_example(item: Dict[str, Any], index: int, split_name: str) -> Dict[str, str]:
    example = {
        "id": as_text(item.get("id") or item.get("story_id") or f"{split_name}_{index}"),
        "story_id": as_text(item.get("story_id")),
        "nat_premises": as_text(first_present(item, ("nat_premises", "nat_premise", "premises"))),
        "nat_conclusion": as_text(first_present(item, ("nat_conclusion", "conclusion"))),
        "fol_premises": as_text(first_present(item, ("fol_premises", "fol_premise"))),
        "fol_conclusion": as_text(item.get("fol_conclusion")),
        "label": as_text(item.get("label")),
    }
    required = ["nat_premises", "nat_conclusion", "fol_premises", "fol_conclusion"]
    missing = [field for field in required if not example[field]]
    if missing:
        example["_missing"] = ",".join(missing)
    return example


def build_source(example: Dict[str, str]) -> str:
    if USE_INSTRUCTION:
        return (
            f"{INSTRUCTION}\n"
            "Premises:\n"
            f"{example['nat_premises']}\n"
            "Conclusion:\n"
            f"{example['nat_conclusion']}"
        )
    return (
        "Premises:\n"
        f"{example['nat_premises']}\n"
        "Conclusion:\n"
        f"{example['nat_conclusion']}"
    )


def build_target(example: Dict[str, str]) -> str:
    return f"{example['fol_premises']}\n{example['fol_conclusion']}".strip()


def validate_supervised_rows(rows: List[Dict[str, str]], split_name: str) -> List[Dict[str, str]]:
    bad_rows = [row for row in rows if row.get("_missing")]
    if bad_rows:
        print(f"Bad {split_name} examples with missing fields:")
        for row in bad_rows[:10]:
            print({"id": row["id"], "missing": row["_missing"]})
        raise ValueError(f"{split_name} has {len(bad_rows)} rows missing required Basic 2 fields.")
    return rows

In [9]:
# ============================================================
# Load official train / valid / test data
# ============================================================

train_raw_all = load_json_list(TRAIN_PATH)

if USE_INTERNAL_SPLIT:
    print("No separate supervised FOL validation file found. Splitting TRAIN_PATH into train/valid.")
    rng = random.Random(SEED)
    indices = list(range(len(train_raw_all)))
    rng.shuffle(indices)

    n_valid = max(1, int(len(indices) * VALID_RATIO))
    valid_idx = set(indices[:n_valid])

    train_raw = [row for i, row in enumerate(train_raw_all) if i not in valid_idx]
    valid_raw = [row for i, row in enumerate(train_raw_all) if i in valid_idx]
else:
    print("Using official train/valid split.")
    train_raw = train_raw_all
    valid_raw = load_json_list(VALID_PATH)

train_examples = [normalize_example(row, i, "train") for i, row in enumerate(train_raw)]
valid_examples = [normalize_example(row, i, "valid") for i, row in enumerate(valid_raw)]

train_examples = validate_supervised_rows(train_examples, "train")
valid_examples = validate_supervised_rows(valid_examples, "valid")

test_examples = []
if TEST_PATH is not None and Path(TEST_PATH).exists():
    test_raw = load_json_list(TEST_PATH)

    # Test may or may not have gold FOL. Do not validate it as supervised.
    test_examples = []
    for i, row in enumerate(test_raw):
        test_examples.append({
            "id": as_text(row.get("id") or row.get("story_id") or f"test_{i}"),
            "story_id": as_text(row.get("story_id")),
            "nat_premises": as_text(first_present(row, ("nat_premises", "nat_premise", "premises"))),
            "nat_conclusion": as_text(first_present(row, ("nat_conclusion", "conclusion"))),
            "fol_premises": as_text(first_present(row, ("fol_premises", "fol_premise"))),
            "fol_conclusion": as_text(row.get("fol_conclusion")),
            "label": as_text(row.get("label")),
        })

print("Number of train rows:", len(train_examples))
print("Number of valid rows:", len(valid_examples))
print("Number of test rows:", len(test_examples))

print("\nFirst source:\n", build_source(train_examples[0]))
print("\nFirst target RAW FOLIO FOL:\n", build_target(train_examples[0]))

if test_examples:
    print("\nFirst test source:\n", build_source(test_examples[0]))

Using official train/valid split.
Number of train rows: 856
Number of valid rows: 93
Number of test rows: 600

First source:
 Translate the following natural language premises and conclusion to first-order logic.
Premises:
All people who regularly drink coffee are dependent on caffeine.People regularly drink coffee, or they don't want to be addicted to caffeine, or both.No one who doesn't want to be addicted to caffeine is unaware that caffeine is a drug.Rina is either a student who is unaware that caffeine is a drug, or she is not a student and is she aware that caffeine is a drug.Rina  is either a student who is dependent on caffeine, or she is not a student and not dependent on caffeine.
Conclusion:
Rina doesn't want to be addicted to caffeine or is unaware that caffeine is a drug.

First target RAW FOLIO FOL:
 ∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))
∀x (DrinkRegularly(x, coffee) ∨ ¬WantToBeAddictedTo(x, caffeine))
∀x (¬WantToBeAddictedTo(x, caffeine) → AwareThat

In [ ]:
## Tokenizer and Dataset

In [10]:
print("Loading tokenizer:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

train_rows = [{"id": row["id"], "source": build_source(row), "target": build_target(row)} for row in train_examples]
valid_rows = [{"id": row["id"], "source": build_source(row), "target": build_target(row)} for row in valid_examples]
raw_train_dataset = Dataset.from_list(train_rows)
raw_valid_dataset = Dataset.from_list(valid_rows)
print("Train rows:", len(train_rows))
print("Valid rows:", len(valid_rows))
print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)

Loading tokenizer: /kaggle/input/datasets/trnlnhtho/mt5-base/mt5-base
Train rows: 856
Valid rows: 93
pad_token: <pad> 0


In [11]:
def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, Any]:
    model_inputs = tokenizer(
        batch["source"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        batch["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    label_ids = labels["input_ids"]
    pad_id = tokenizer.pad_token_id
    if pad_id is not None:
        label_ids = [[tok if tok != pad_id else -100 for tok in seq] for seq in label_ids]
    model_inputs["labels"] = label_ids
    return model_inputs

train_dataset = raw_train_dataset.map(tokenize_batch, batched=True, remove_columns=["id", "source", "target"])
valid_dataset = raw_valid_dataset.map(tokenize_batch, batched=True, remove_columns=["id", "source", "target"])


def check_tokenized_dataset(ds, name: str):
    input_lens, label_lens, non_ignored_counts, bad = [], [], [], []
    for i in range(len(ds)):
        row = ds[i]
        input_lens.append(len(row["input_ids"]))
        label_lens.append(len(row["labels"]))
        non_ignored = sum(1 for token in row["labels"] if token != -100)
        non_ignored_counts.append(non_ignored)
        if non_ignored == 0:
            bad.append(i)
    print(f"\n{name} tokenized dataset")
    print("num rows:", len(ds))
    print("input len min/max/mean:", min(input_lens), max(input_lens), float(np.mean(input_lens)))
    print("label len min/max/mean:", min(label_lens), max(label_lens), float(np.mean(label_lens)))
    print("non-ignored labels min/max/mean:", min(non_ignored_counts), max(non_ignored_counts), float(np.mean(non_ignored_counts)))
    print("all-ignored label rows:", len(bad))
    if bad:
        raise ValueError(f"{name} has rows with labels all set to -100.")

check_tokenized_dataset(train_dataset, "train")
check_tokenized_dataset(valid_dataset, "valid")
print(train_dataset)
print(valid_dataset)

Map:   0%|          | 0/856 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]


train tokenized dataset
num rows: 856
input len min/max/mean: 41 366 125.11098130841121
label len min/max/mean: 26 512 153.5
non-ignored labels min/max/mean: 26 512 153.5
all-ignored label rows: 0

valid tokenized dataset
num rows: 93
input len min/max/mean: 41 252 126.51612903225806
label len min/max/mean: 47 355 167.51612903225808
non-ignored labels min/max/mean: 47 355 167.51612903225808
all-ignored label rows: 0
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 856
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 93
})


In [12]:
def load_mt5_base_model():
    config = AutoConfig.from_pretrained(MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
    config.tie_word_embeddings = False
    base = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        config=config,
        local_files_only=LOCAL_FILES_ONLY,
    )
    if base.config.pad_token_id is None and tokenizer.pad_token_id is not None:
        base.config.pad_token_id = tokenizer.pad_token_id
    if base.config.decoder_start_token_id is None and base.config.pad_token_id is not None:
        base.config.decoder_start_token_id = base.config.pad_token_id
    return base


def check_lm_head_state(model):
    if hasattr(model, "shared") and hasattr(model, "lm_head"):
        same_object = model.shared.weight.data_ptr() == model.lm_head.weight.data_ptr()
        max_abs_diff = (model.shared.weight.detach().cpu() - model.lm_head.weight.detach().cpu()).abs().max().item()
        print("lm_head vs shared same_object:", same_object)
        print("lm_head vs shared max_abs_diff:", max_abs_diff)
        print("config.tie_word_embeddings:", model.config.tie_word_embeddings)

print("Loading base model:", MODEL_NAME)
base_model = load_mt5_base_model()
check_lm_head_state(base_model)

if ENABLE_GRADIENT_CHECKPOINTING:
    base_model.gradient_checkpointing_enable()
    base_model.config.use_cache = False
    if hasattr(base_model, "enable_input_require_grads"):
        base_model.enable_input_require_grads()

if USE_LORA:
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
else:
    model = base_model
    for p in model.parameters():
        p.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print("Total params:", total_params)
print("Trainable params:", trainable_params)
print("Trainable percentage:", 100 * trainable_params / total_params)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
print("Model device:", next(model.parameters()).device)

Loading base model: /kaggle/input/datasets/trnlnhtho/mt5-base/mt5-base


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

lm_head vs shared same_object: False
lm_head vs shared max_abs_diff: 113.6005859375
config.tie_word_embeddings: False
trainable params: 7,077,888 || all params: 973,651,200 || trainable%: 0.7269
Total params: 973651200
Trainable params: 7077888
Trainable percentage: 0.7269428723551103
Model device: cuda:0


In [13]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
features = [train_dataset[i] for i in range(min(2, len(train_dataset)))]
batch = collator(features)
batch = {key: value.to(DEVICE) for key, value in batch.items()}

model.eval()
with torch.no_grad():
    outputs = model(**batch)
loss = outputs.loss
one_batch_loss = float(loss.detach().cpu())
print("One-batch forward loss:", one_batch_loss)
print("Loss finite:", bool(torch.isfinite(loss).item()))
if not torch.isfinite(loss).item():
    raise ValueError("One-batch loss is NaN or Inf.")
if one_batch_loss > 200:
    raise ValueError(f"One-batch loss is unexpectedly high ({one_batch_loss}).")
model.train()
print("One-batch finite-loss test passed.")

One-batch forward loss: 13.985503196716309
Loss finite: True
One-batch finite-loss test passed.


In [14]:
def make_training_args(output_dir: str, num_train_epochs: int) -> Seq2SeqTrainingArguments:
    signature = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    candidates = {
        "output_dir": output_dir,
        "overwrite_output_dir": True,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "num_train_epochs": num_train_epochs,
        "learning_rate": LEARNING_RATE,
        "optim": OPTIM,
        "eval_steps": EVAL_STEPS,
        "save_steps": SAVE_STEPS,
        "logging_steps": LOGGING_STEPS,
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "report_to": "none",
        "predict_with_generate": True,
        "generation_max_length": MAX_TARGET_LENGTH,
        "fp16": USE_FP16,
        "bf16": USE_BF16,
        "logging_nan_inf_filter": False,
        "max_grad_norm": 1.0,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "save_strategy": "steps",
        "seed": SEED,
        "dataloader_pin_memory": False,
    }
    if "eval_strategy" in signature:
        candidates["eval_strategy"] = "steps"
    elif "evaluation_strategy" in signature:
        candidates["evaluation_strategy"] = "steps"
    supported = {key: value for key, value in candidates.items() if key in signature}
    dropped = sorted(set(candidates) - set(supported))
    if dropped:
        print("Dropped unsupported Seq2SeqTrainingArguments keys:", dropped)
    return Seq2SeqTrainingArguments(**supported)

num_epochs = NUM_EPOCHS_DEBUG if RUN_MODE == "debug" else NUM_EPOCHS_FULL
training_args = make_training_args(OUTPUT_DIR, num_epochs)
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": valid_dataset,
    "data_collator": collator,
}
trainer_signature = inspect.signature(Seq2SeqTrainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Seq2SeqTrainer(**trainer_kwargs)
print("Seq2SeqTrainer created.")
print("Output dir:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print("LR:", LEARNING_RATE)

Dropped unsupported Seq2SeqTrainingArguments keys: ['overwrite_output_dir']
Seq2SeqTrainer created.
Output dir: /kaggle/working/outputs/basic2_mt5base_raw_folio_official_lora_qkvo_r32
Epochs: 10
LR: 3e-05


In [15]:
print("Starting Basic 2 mT5 raw FOLIO FOL training.")
train_result = trainer.train()
train_metrics = train_result.metrics
print("Train metrics:", train_metrics)

for key, value in train_metrics.items():
    if isinstance(value, (float, int)) and not math.isfinite(float(value)):
        raise ValueError(f"Training metric {key} is not finite: {value}")

eval_metrics = trainer.evaluate()
print("Eval metrics:", eval_metrics)
if "eval_loss" not in eval_metrics:
    raise ValueError("eval_loss missing from evaluation metrics.")
if not math.isfinite(float(eval_metrics["eval_loss"])):
    raise ValueError(f"eval_loss is not finite: {eval_metrics['eval_loss']}")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.save_state()

with open(Path(OUTPUT_DIR) / "basic2_train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(train_metrics, f, ensure_ascii=False, indent=2)
with open(Path(OUTPUT_DIR) / "basic2_eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, ensure_ascii=False, indent=2)

adapter_config = Path(OUTPUT_DIR) / "adapter_config.json"
adapter_safetensors = Path(OUTPUT_DIR) / "adapter_model.safetensors"
adapter_bin = Path(OUTPUT_DIR) / "adapter_model.bin"
print("adapter_config exists:", adapter_config.exists())
print("adapter_model.safetensors exists:", adapter_safetensors.exists())
print("adapter_model.bin exists:", adapter_bin.exists())
if USE_LORA and (not adapter_config.exists() or not (adapter_safetensors.exists() or adapter_bin.exists())):
    raise FileNotFoundError("LoRA adapter files were not saved as expected.")
print("Basic 2 checkpoint saved to:", OUTPUT_DIR)

Starting Basic 2 mT5 raw FOLIO FOL training.


Step,Training Loss,Validation Loss
25,70.850775,13.145786
50,67.194012,12.724031
75,66.059711,12.425708
100,62.596533,11.654523
125,64.532715,10.936441
150,60.629596,10.374872
175,56.505438,9.643469
200,54.579761,8.986906
225,43.744714,8.205152
250,43.004645,7.653852


Train metrics: {'train_runtime': 633.4124, 'train_samples_per_second': 13.514, 'train_steps_per_second': 3.379, 'total_flos': 2553524241868800.0, 'train_loss': 19.082209396362305, 'epoch': 10.0}


Eval metrics: {'eval_loss': 1.5529505014419556, 'eval_runtime': 1.5314, 'eval_samples_per_second': 60.728, 'eval_steps_per_second': 60.728, 'epoch': 10.0}
adapter_config exists: True
adapter_model.safetensors exists: True
adapter_model.bin exists: False
Basic 2 checkpoint saved to: /kaggle/working/outputs/basic2_mt5base_raw_folio_official_lora_qkvo_r32


In [ ]:
# def build_bad_words_ids(tokenizer):
#     bad_words_ids = []
#     for tok in getattr(tokenizer, "additional_special_tokens", []):
#         if tok.startswith("<extra_id_"):
#             ids = tokenizer.encode(tok, add_special_tokens=False)
#             if ids:
#                 bad_words_ids.append(ids)
#     return bad_words_ids


# def clean_generated_fol(text: str) -> str:
#     text = text or ""

#     for i in range(200):
#         text = text.replace(f"<extra_id_{i}>", "")

#     text = text.replace("<pad>", "")
#     text = text.replace("</s>", "")
#     text = " ".join(text.split()).strip()

#     tokens = text.split()
#     if len(tokens) > 160:
#         tokens = tokens[:160]

#     text = " ".join(tokens).strip()

#     if not text:
#         return "# EMPTY_MT5_GENERATION"

#     bad_patterns = [
#         "x x x x",
#         "x(x) x(x)",
#         "citizen of Lawton Park, citizen of Lawton Park",
#     ]

#     if any(pattern in text for pattern in bad_patterns):
#         return "# INVALID_MT5_GENERATION\n" + text[:700]

#     return text


# def load_inference_model(adapter_dir: str):
#     print("Loading inference base model:", MODEL_NAME)

#     base = load_mt5_base_model()
#     check_lm_head_state(base)

#     if USE_LORA:
#         inf_model = PeftModel.from_pretrained(base, adapter_dir)
#     else:
#         inf_model = base

#     inf_model.to(DEVICE)
#     inf_model.eval()

#     print("Inference model loaded from:", adapter_dir)
#     return inf_model


# infer_model = load_inference_model(OUTPUT_DIR)


# def generate_mt5_fol(source: str, max_new_tokens: int = GEN_MAX_NEW_TOKENS) -> Dict[str, str]:
#     inputs = tokenizer(
#         source,
#         return_tensors="pt",
#         max_length=MAX_SOURCE_LENGTH,
#         truncation=True,
#     )
#     inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

#     bad_words_ids = build_bad_words_ids(tokenizer)

#     with torch.no_grad():
#         output_ids = infer_model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             num_beams=4,
#             do_sample=False,
#             no_repeat_ngram_size=3,
#             repetition_penalty=1.35,
#             length_penalty=0.8,
#             early_stopping=True,
#             bad_words_ids=bad_words_ids if bad_words_ids else None,
#             pad_token_id=tokenizer.pad_token_id,
#             eos_token_id=tokenizer.eos_token_id,
#         )

#     raw = tokenizer.decode(output_ids[0], skip_special_tokens=False).strip()
#     clean = clean_generated_fol(raw)

#     return {
#         "raw_generated_fol": raw,
#         "generated_fol": clean,
#     }


# print("Generation utilities ready.")

In [ ]:
# if RUN_MODE == "overfit":
#     sample_source_rows = train_rows[:10]
#     print("Sampling from overfit train rows.")
# else:
#     sample_source_rows = valid_rows[:10]
#     print("Sampling from validation rows.")

# sample_outputs = []

# for row in sample_source_rows:
#     source = row["source"]
#     gold = row["target"]
#     pred = generate_mt5_fol(source)

#     item = {
#         "id": row["id"],
#         "input": source,
#         "gold_fol": gold,
#         "raw_generated_fol": pred["raw_generated_fol"],
#         "generated_fol": pred["generated_fol"],
#     }
#     sample_outputs.append(item)

#     print("=" * 80)
#     print("ID:", row["id"])
#     print("GOLD_FOL:\\n", gold)
#     print("RAW_GENERATED_FOL:\\n", pred["raw_generated_fol"])
#     print("CLEAN_GENERATED_FOL:\\n", pred["generated_fol"])

# sample_path = Path(OUTPUT_DIR) / "sample_generations.json"
# with open(sample_path, "w", encoding="utf-8") as f:
#     json.dump(sample_outputs, f, ensure_ascii=False, indent=2)

# print("Saved samples to:", sample_path)